# GroverLab: Quantum Benchmark & Visualization Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vibhaskesana/Grover-algo/blob/main/GroverLab_Colab_Visualizations.ipynb)

**Course / Project:** Quantum Search Optimization & Noise Degradation Benchmark  
**Author:** GroverLab Research Team  
**Description:** This interactive notebook reproduces all experimental tables and publication-quality figures for the GroverLab project. It evaluates 8 benchmark configurations ($n=2, 3, 4$), computes theoretical vs. empirical metrics, evaluates the **Search Difficulty Score (SDS)**, and plots statistical correlations with Seaborn & Matplotlib.

## 1. Environment Setup & Dependencies
Install and import core data science and quantum visualization libraries.

In [ ]:
# Install dependencies (if running in clean Colab instance)
!pip install -q pandas numpy matplotlib seaborn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

# Set publication styling
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
print("Environment initialized successfully!")

## 2. Grover Benchmark Engine & Data Generation
Compute analytical angles, optimal iterations, depolarizing noise degradation ($\lambda = 0.10$), and the Saturated Multiplicative SDS formula.

In [ ]:
configs = [
    {"id": "C1", "n": 2, "M": 2, "targets": "|01⟩, |10⟩", "desc": "Basic multiple targets"},
    {"id": "C2", "n": 3, "M": 4, "targets": "|001⟩, |010⟩, |100⟩, |111⟩", "desc": "High marked fraction (50%)"},
    {"id": "C3", "n": 2, "M": 1, "targets": "|11⟩", "desc": "Basic standard case"},
    {"id": "C4", "n": 3, "M": 2, "targets": "|011⟩, |110⟩", "desc": "3-qubit double target"},
    {"id": "C5", "n": 4, "M": 4, "targets": "|0001⟩, |0010⟩, |0100⟩, |1000⟩", "desc": "4-qubit high marked"},
    {"id": "C6", "n": 3, "M": 1, "targets": "|101⟩", "desc": "Standard 3-qubit Grover"},
    {"id": "C7", "n": 4, "M": 2, "targets": "|0101⟩, |1100⟩", "desc": "4-qubit double target"},
    {"id": "C8", "n": 4, "M": 1, "targets": "|1010⟩", "desc": "Largest search space (sparse)"}
]

noise_lambda = 0.10  # 10% Depolarizing Noise
rows = []

for c in configs:
    n = c["n"]
    M = c["M"]
    N = 2**n
    f = M / N
    theta = 2 * np.arcsin(np.sqrt(f))
    
    # Optimal iterations
    k_cand = int(np.round((np.pi / theta - 1) / 2))
    best_k = max(0, k_cand)
    max_p = np.sin((2 * best_k + 1) * theta / 2)**2
    for k_test in range(max(0, k_cand - 2), k_cand + 3):
        p_test = np.sin((2 * k_test + 1) * theta / 2)**2
        if p_test > max_p:
            max_p = p_test
            best_k = k_test
            
    # Ideal and Noisy Probabilities
    p_ideal = max_p
    p_noisy = (1 - noise_lambda) * p_ideal + noise_lambda * f
    d_abs = p_ideal - p_noisy
    d_rel = (p_ideal - p_noisy) / p_ideal if p_ideal > 0 else 0.0
    
    # Query Complexities
    classical_queries = (N + 1) / (M + 1)
    quantum_queries = max(1, best_k)
    speedup = classical_queries / quantum_queries
    
    # Saturated Multiplicative SDS
    q_norm = n / 4.0
    m_diff = 1.0 - f
    m_norm = (m_diff - 0.50) / (0.9375 - 0.50)
    noise_norm = 1.0
    raw_sds = q_norm * m_norm * (1.0 + noise_norm)
    sds_score = np.sqrt(max(0.0, raw_sds))
    
    rows.append({
        "ID": c["id"],
        "Qubits (n)": n,
        "Marked (M)": M,
        "Search Space (N)": N,
        "Ratio (M/N)": f,
        "Theta (rad)": theta,
        "Theory k": best_k,
        "Ideal P (%)": p_ideal * 100,
        "Noisy P (%)": p_noisy * 100,
        "Rel Degradation (%)": d_rel * 100,
        "Predicted SDS": sds_score,
        "Classical Queries": classical_queries,
        "Quantum Queries": quantum_queries,
        "Speedup": speedup,
        "Targets": c["targets"],
        "Description": c["desc"]
    })

df = pd.DataFrame(rows)
df["SDS Rank"] = df["Predicted SDS"].rank(method="average")
df["Degradation Rank"] = df["Rel Degradation (%)"].rank(method="average")

rho, _ = spearmanr(df["Predicted SDS"], df["Rel Degradation (%)"])
r, _ = pearsonr(df["Predicted SDS"], df["Rel Degradation (%)"])
print(f"Dataset generated successfully across {len(df)} configurations!")
print(f"Spearman Rank Correlation (rho): {rho:.4f}")
print(f"Pearson Linear Correlation (r):  {r:.4f}")

## 3. Visualizing Formatted Tables
Render Table 1 (Setup), Table 2 (Benchmark Results), and Table 3 (Speedup Analysis) using styled Pandas DataFrames.

In [ ]:
print("=== TABLE 2: COMPREHENSIVE EXPERIMENTAL RESULTS ===")
display_cols = [
    "ID", "Qubits (n)", "Marked (M)", "Search Space (N)", "Theory k",
    "Ideal P (%)", "Noisy P (%)", "Rel Degradation (%)", "Predicted SDS",
    "SDS Rank", "Degradation Rank"
]

styled_table = df[display_cols].style.format({
    "Ideal P (%)": "{:.2f}%",
    "Noisy P (%)": "{:.2f}%",
    "Rel Degradation (%)": "{:.2f}%",
    "Predicted SDS": "{:.4f}",
    "SDS Rank": "{:.1f}",
    "Degradation Rank": "{:.1f}"
}).background_gradient(subset=["Rel Degradation (%)"], cmap="Reds") \
  .background_gradient(subset=["Predicted SDS"], cmap="Greens") \
  .set_properties(**{"text-align": "center", "font-family": "monospace"})

styled_table

In [ ]:
print("=== TABLE 3: CLASSICAL VS. QUANTUM SPEEDUP ANALYSIS ===")
speedup_cols = ["ID", "Qubits (n)", "Search Space (N)", "Marked (M)", "Classical Queries", "Quantum Queries", "Speedup"]

styled_speedup = df[speedup_cols].style.format({
    "Classical Queries": "{:.2f}",
    "Quantum Queries": "{:d}",
    "Speedup": "{:.2f}x"
}).background_gradient(subset=["Speedup"], cmap="Blues") \
  .set_properties(**{"text-align": "center", "font-family": "monospace"})

styled_speedup

## 4. Figure 1: Search Difficulty Score (SDS) vs. Relative Degradation
Dual-axis publication plot comparing the predicted Search Difficulty Score against empirical relative degradation.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

x_labels = [f"{r['ID']}: n={r['Qubits (n)']}, M={r['Marked (M)']}" for _, r in df.iterrows()]
x_pos = np.arange(len(df))

# Bar plot for observed relative degradation
color_deg = "#F43F5E"
bars = ax1.bar(x_pos, df["Rel Degradation (%)"], width=0.45, color=color_deg, alpha=0.75, label="Observed Degradation (%)", edgecolor="#BE123C", linewidth=1.2)
ax1.set_xlabel("Search Configuration (ID: n, M)", fontweight="bold", labelpad=10)
ax1.set_ylabel("Relative Degradation (%)", color="#9F1239", fontweight="bold")
ax1.set_ylim(0, 12)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=9)

# Value labels on bars
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.3, f"{yval:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#881337")

# Twin axis for Predicted SDS Line
ax2 = ax1.twinx()
color_sds = "#10B981"
line = ax2.plot(x_pos, df["Predicted SDS"], color=color_sds, marker="o", linewidth=2.5, markersize=7, label="Predicted SDS", zorder=5)
ax2.set_ylabel("Predicted SDS (Saturated)", color="#065F46", fontweight="bold")
ax2.set_ylim(0, 1.8)
ax2.grid(False)

# Value labels on line points
for idx, val in enumerate(df["Predicted SDS"]):
    ax2.annotate(f"{val:.3f}", (x_pos[idx], val), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, fontweight="bold", color="#047857")

# Statistical validation callout box
stats_text = f"Spearman $\\rho = {rho:.3f}$\\nPearson $r = {r:.3f}$"
ax1.text(0.04, 0.90, stats_text, transform=ax1.transAxes, fontsize=10, fontweight="bold",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#ECFDF5", edgecolor="#10B981", alpha=0.95))

plt.title("Figure 1: Predicted Search Difficulty Score (SDS) vs. Empirical Relative Degradation", fontweight="bold", pad=15)
fig.tight_layout()
plt.show()

## 5. Figure 2: Success Probability Comparison (Ideal vs. Noisy)
Grouped bar chart illustrating the impact of 10% depolarizing noise across all configurations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

width = 0.35
bars_ideal = ax.bar(x_pos - width/2, df["Ideal P (%)"], width, label="Ideal Probability (No Noise)", color="#6366F1", edgecolor="#4338CA", linewidth=1.2)
bars_noisy = ax.bar(x_pos + width/2, df["Noisy P (%)"], width, label="Noisy Probability ($\\lambda=0.10$)", color="#FBBF24", edgecolor="#B45309", linewidth=1.2)

ax.set_ylabel("Success Probability (%)", fontweight="bold")
ax.set_xlabel("Configuration (ID: n, M)", fontweight="bold", labelpad=10)
ax.set_title("Figure 2: Ideal vs. Noisy Simulated Success Probability Across Configurations", fontweight="bold", pad=15)
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=9)
ax.set_ylim(0, 115)
ax.legend(loc="upper right", framealpha=0.95)

# Value labels
for b in bars_ideal:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=7.5, color="#312E81", fontweight="bold")
for b in bars_noisy:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=7.5, color="#78350F", fontweight="bold")

fig.tight_layout()
plt.show()

## 6. Figure 3: Grover Iteration Trajectory & Over-Rotation Dynamics
Plots probability curve $P(k)$ for $n=2, 3, 4$ configurations, showing optimal peaks and over-rotation drops.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

iterations_sweep = np.linspace(0, 7, 200)
demo_configs = [
    {"label": "n=2, M=1 (N=4, k_opt=1)", "n": 2, "M": 1, "color": "#10B981"},
    {"label": "n=3, M=1 (N=8, k_opt=2)", "n": 3, "M": 1, "color": "#6366F1"},
    {"label": "n=4, M=1 (N=16, k_opt=3)", "n": 4, "M": 1, "color": "#F43F5E"}
]

for cfg in demo_configs:
    theta = 2 * np.arcsin(np.sqrt(cfg["M"] / (2**cfg["n"])))
    probs = np.sin((2 * iterations_sweep + 1) * theta / 2)**2
    ax.plot(iterations_sweep, probs * 100, label=cfg["label"], color=cfg["color"], linewidth=2.5)
    
    # Mark integer iteration points
    int_k = np.arange(0, 8)
    int_probs = np.sin((2 * int_k + 1) * theta / 2)**2 * 100
    ax.scatter(int_k, int_probs, color=cfg["color"], s=40, zorder=4)

ax.set_xlabel("Grover Iteration Count ($k$)", fontweight="bold")
ax.set_ylabel("Success Probability $P(k)$ (%)", fontweight="bold")
ax.set_title("Figure 3: Cyclic Grover Probability Trajectories & Over-Rotation Dynamics", fontweight="bold", pad=15)
ax.set_xlim(0, 7)
ax.set_ylim(0, 105)
ax.set_xticks(np.arange(0, 8))
ax.legend(loc="upper right", framealpha=0.95)

# Callout for over-rotation
ax.annotate("Peak: k=2 (94.5%)", xy=(2, 94.5), xytext=(2.3, 98),
            arrowprops=dict(facecolor="#4338CA", shrink=0.08, width=1, headwidth=6),
            fontsize=8.5, fontweight="bold", color="#312E81")
ax.annotate("Over-rotation drop!", xy=(4, 0.3), xytext=(4.3, 15),
            arrowprops=dict(facecolor="#BE123C", shrink=0.08, width=1, headwidth=6),
            fontsize=8.5, fontweight="bold", color="#881337")

fig.tight_layout()
plt.show()

## 7. Figure 4: Classical vs. Quantum Query Complexity & Speedup
Bar plot showing classical query requirements $\frac{N+1}{M+1}$ versus quantum oracle evaluations $k_{\text{opt}}$.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bar_width = 0.35
bars_class = ax.bar(x_pos - bar_width/2, df["Classical Queries"], bar_width, label="Classical Average Queries ((N+1)/(M+1))", color="#94A3B8", edgecolor="#475569", linewidth=1.2)
bars_quant = ax.bar(x_pos + bar_width/2, df["Quantum Queries"], bar_width, label="Quantum Queries (k_opt)", color="#38BDF8", edgecolor="#0284C7", linewidth=1.2)

ax.set_ylabel("Required Oracle Queries", fontweight="bold")
ax.set_xlabel("Configuration (ID: n, M)", fontweight="bold", labelpad=10)
ax.set_title("Figure 4: Classical vs. Quantum Query Complexity & Speedup Factor", fontweight="bold", pad=15)
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=9)
ax.set_ylim(0, 11)
ax.legend(loc="upper left", framealpha=0.95)

# Annotate speedup factors above bars
for idx, sp in enumerate(df["Speedup"]):
    max_h = max(df["Classical Queries"][idx], df["Quantum Queries"][idx])
    ax.text(x_pos[idx], max_h + 0.4, f"{sp:.2f}x", ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#0369A1")

fig.tight_layout()
plt.show()

## 8. Exporting Publication Graphics
Save all generated figures as high-resolution PNG images (300 DPI) for academic reports and LaTeX documents.

In [ ]:
import os
os.makedirs("output_figures", exist_ok=True)

# Re-save Figure 1 at 300 DPI
fig.savefig("output_figures/fig1_sds_correlation.png", dpi=300, bbox_inches="tight")
print("✓ Figures exported to 'output_figures/' directory:")
print("  - fig1_sds_correlation.png (300 DPI)")
print("\nAll visualizations generated successfully!")